In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
import warnings
import time

In [3]:
warnings.filterwarnings('ignore')

In [4]:
data = pd.read_csv('vehicle_predictive_maintenance_dataset.csv')

In [5]:
print("Dataset shape:", data.shape)
print("\nMissing values:")
print(data.isnull().sum())

Dataset shape: (100000, 7)

Missing values:
mileage         0
engine_temp     0
oil_pressure    0
vibration       0
brake_wear      0
rpm_variance    0
failure         0
dtype: int64


In [6]:
X = data.drop(['failure'], axis=1)
y = data['failure']

In [7]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

In [8]:
selector = SelectKBest(score_func=f_classif, k='all')
selector.fit(X_scaled, y)

,"score_func score_func: callable, default=f_classifFunction taking two arrays X and y, and returning a pair of arrays(scores, pvalues) or a single array with scores.Default is f_classif (see below ""See Also""). The default function onlyworks with classification tasks... versionadded:: 0.18",<function f_c...00264F2A5ACA0>
,"k k: int or ""all"", default=10Number of top features to select.The ""all"" option bypasses selection, for use in a parameter search.",'all'


In [9]:
feature_scores = pd.DataFrame({
    'Feature': X.columns,
    'Score': selector.scores_
}).sort_values('Score', ascending=False)

print("\nFeature Importance Scores:")
print(feature_scores)

# Select top features
top_features = feature_scores.head(6)['Feature'].values
X_selected = X_scaled[top_features]

print(f"\nSelected features: {list(top_features)}")


Feature Importance Scores:
        Feature      Score
4    brake_wear  90.232322
1   engine_temp  54.322581
3     vibration  10.609509
2  oil_pressure   2.093480
5  rpm_variance   0.842495
0       mileage   0.056238

Selected features: ['brake_wear', 'engine_temp', 'vibration', 'oil_pressure', 'rpm_variance', 'mileage']


In [10]:
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.3, random_state=42, stratify=y)
print(f"\nTraining set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")


Training set size: (70000, 6)
Test set size: (30000, 6)


In [11]:
models = {
    'GradientBoosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {
            'n_estimators': [100, 200],
            'learning_rate': [0.05, 0.1],
            'max_depth': [3, 4],
            'subsample': [0.8, 1.0]
        }
    },

    'RandomForest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [100, 200],
            'max_depth': [10, None],
            'min_samples_split': [2, 5],
            'max_features': ['sqrt']
        }
    },

    'SVM': {
         'model': SVC(probability=True, random_state=42),
    'params': {
        'C': [0.1, 1, 10],
        'kernel': ['linear']
    }
}
        }



In [12]:
# Train and evaluate models
best_models = {}
best_scores = {}

In [13]:
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV

for name, config in models.items():
    print(f"\n{'='*50}")
    print(f"Training {name} with HalvingGridSearchCV...")
    start_time = time.time()

    halving_search = HalvingGridSearchCV(
        config['model'],
        config['params'],
        cv=3,
        scoring='accuracy',
        n_jobs=-1,
        factor=3,
        verbose=1,
        random_state=42
    )
    halving_search.fit(X_train, y_train)
    
    best_models[name] = halving_search.best_estimator_
    best_scores[name] = halving_search.best_score_
    
    end_time = time.time()
    print(f"Training time: {end_time - start_time:.2f} seconds")
    print(f"Best parameters for {name}: {halving_search.best_params_}")
    print(f"Best cross-validation score for {name}: {halving_search.best_score_:.4f}")


Training GradientBoosting with HalvingGridSearchCV...
n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 3
min_resources_: 7777
max_resources_: 70000
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 16
n_resources: 7777
Fitting 3 folds for each of 16 candidates, totalling 48 fits
----------
iter: 1
n_candidates: 6
n_resources: 23331
Fitting 3 folds for each of 6 candidates, totalling 18 fits
----------
iter: 2
n_candidates: 2
n_resources: 69993
Fitting 3 folds for each of 2 candidates, totalling 6 fits
Training time: 43.07 seconds
Best parameters for GradientBoosting: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}
Best cross-validation score for GradientBoosting: 0.8949

Training RandomForest with HalvingGridSearchCV...
n_iterations: 2
n_required_iterations: 2
n_possible_iterations: 2
min_resources_: 23333
max_resources_: 70000
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 8
n_resources: 23

In [14]:
# Find the best performing model
best_model_name = max(best_scores, key=best_scores.get)
best_model = best_models[best_model_name]

In [15]:
print(f"\n{'='*50}")
print(f"BEST MODEL: {best_model_name}")
print(f"Best Cross-validation Score: {best_scores[best_model_name]:.4f}")


BEST MODEL: SVM
Best Cross-validation Score: 0.8950


In [16]:
# Final evaluation on test set
y_pred_best = best_model.predict(X_test)
final_accuracy = accuracy_score(y_test, y_pred_best)
print(f"\nFINAL TEST ACCURACY: {final_accuracy:.4f}")


FINAL TEST ACCURACY: 0.8950


In [17]:
# Detailed classification report
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred_best))


Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.89      1.00      0.94     26849
           1       0.00      0.00      0.00      3151

    accuracy                           0.89     30000
   macro avg       0.45      0.50      0.47     30000
weighted avg       0.80      0.89      0.85     30000



In [18]:
# Confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_best))

Confusion Matrix:
[[26849     0]
 [ 3151     0]]


Beacuse SVM took 294.67 seconds to train in linear not (rbf) and,
Gradient Boosting Classifier took only 43.07 seconds both differs their accuracy by 0.001 hence decided to use Gradient Boosting Classifier model, because the data is large